In [1]:
import os 
import re
import time
import torch
import pathlib 
import numpy as np
import pandas as pd
import seaborn as sns
from math import ceil
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader, random_split

from basic_cnn import*

In [2]:
# emb1 = torch.load('/home/chyhsu/Documents/embeddings/sub-105046731194_encoder_embeddings.pt')
# emb2 = torch.load('/home/chyhsu/Documents/embeddings/sub-388018649913_encoder_embeddings.pt')
subject_meta = pd.read_csv('/home/chyhsu/Documents/val_labels/participants.tsv', sep='\t')
output_dir = '/home/chyhsu/Documents/embeddings/'
#output_dir = '/home/chyhsu/Documents/embeddings_sub/'

In [3]:
max_dim0 = 0
max_dim1 = 0
file_shapes = {}
for file in os.listdir(output_dir):
    if file.endswith(".pt"):
        path = os.path.join(output_dir, file)
        emb = torch.load(path)
    if isinstance(emb, torch.Tensor):
        shape = emb.shape
    else:
        shape = emb["embedding"].shape
        file_shapes[file] = shape
    if len(shape) >= 2:
        max_dim0 = max(max_dim0, shape[0])
        max_dim1 = max(max_dim1, shape[1])
    elif len(shape) == 1:
        max_dim0 = max(max_dim0, shape[0])
print("Shapes per file:")
for f, s in file_shapes.items():
    print(f"{f}: {s}")
    print("\nMaximum dimensions across all embeddings:")
    print("max_dim0:", max_dim0)
    print("max_dim1:", max_dim1)

/tmp/ipykernel_269833/1513839757.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  emb = torch.load(path)


Shapes per file:


In [4]:
max_h = max_dim0
max_w = max_dim1
def round_up(x, base=16):
    return int(base * ceil(x / base))
pad_h = round_up(max_h, 16)
pad_w = round_up(max_w, 16)
print("Padded target shape:", pad_h, pad_w)

Padded target shape: 816 768


In [5]:
output_folder = "/home/chyhsu/Documents/padded_embeddings/"
os.makedirs(output_folder, exist_ok=True)
for file in os.listdir(output_dir):
    if file.endswith(".pt"):
        path = os.path.join(output_dir, file)
        emb = torch.load(path)
    if isinstance(emb, dict):
        emb = emb["embedding"]
    h, w = emb.shape
    pad_bottom = pad_h - h
    pad_right = pad_w - w
    padded = F.pad(
    emb,
    (0, pad_right, 0, pad_bottom),
    mode="constant",
    value=0
    )
    save_path = os.path.join(output_folder, file)
    torch.save(padded, save_path)

/tmp/ipykernel_269833/439870433.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  emb = torch.load(path)


In [6]:
sum_ = 0.0
sq_sum = 0.0
count = 0
for f in os.listdir(output_folder):
    if f.endswith(".pt"):
        x = torch.load(os.path.join(output_folder, f)).float()
        sum_ += x.sum().item()
        sq_sum += (x**2).sum().item()
        count += x.numel()
mean = sum_ / count
std = ((sq_sum / count) - mean ** 2) ** 0.5
print("mean:", mean)
print("std:", std)

/tmp/ipykernel_269833/1569208634.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  x = torch.load(os.path.join(output_folder, f)).float()


mean: -0.0032744449582103726
std: 0.05714211522967497


In [7]:
path = pathlib.Path(output_folder)
data_list = []

filename_pattern = re.compile(r'sub-([^_]+)')
for file in path.glob('*.pt'):
    match = filename_pattern.search(file.name)
    subj_id = match.group(1)
    emb_tensor = torch.load(file, map_location='cpu')

    emb_tensor_np = emb_tensor.detach().cpu()
    data_list.append({
        'subject id': subj_id, 
        'embedding': emb_tensor_np
    })

/tmp/ipykernel_269833/1778848341.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  emb_tensor = torch.load(file, map_location='cpu')


In [8]:
emb_df = pd.DataFrame(data_list)
subject_meta['participant_id'] =  subject_meta['participant_id'].astype(str)
age_embedding_df = pd.merge(emb_df, subject_meta, left_on='subject id', right_on='participant_id', how='inner')

In [9]:
age_embedding_df

,subject id,embedding,participant_id,study,sex,age,site,diagnosis,tiv,csfv,gmv,wmv,magnetic_field_strength,acquisition_setting,split
0,447055124363,"[[tensor(0.0002, dtype=torch.bfloat16), tensor...",447055124363,3,female,12.000000,65.0,control,1299.046753,197.279827,670.546616,430.855839,3.0,1.0,internal_test
1,998587653727,"[[tensor(0.0145, dtype=torch.bfloat16), tensor...",998587653727,5,male,45.859001,36.0,control,1584.481992,319.352957,654.915781,606.576325,1.5,1.0,external_test
2,779735428129,"[[tensor(-0.0043, dtype=torch.bfloat16), tenso...",779735428129,4,male,19.000000,3.0,control,1437.500680,249.839705,701.255024,486.206829,3.0,1.0,internal_test
3,993812886852,"[[tensor(-0.0021, dtype=torch.bfloat16), tenso...",993812886852,4,male,21.000000,3.0,control,1626.340516,291.104588,731.531771,602.915369,3.0,1.0,internal_test
4,265646990417,"[[tensor(-0.0025, dtype=torch.bfloat16), tenso...",265646990417,8,female,27.666667,55.0,control,1267.808189,179.824537,613.791798,473.358676,3.0,1.0,external_test
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
747,831135278199,"[[tensor(0.0161, dtype=torch.bfloat16), tensor...",831135278199,5,female,23.110198,36.0,control,1543.829667,243.711308,702.450923,596.331828,1.5,1.0,external_test
748,768859606674,"[[tensor(0.0136, dtype=torch.bfloat16), tensor...",768859606674,2,female,42.760000,20.0,control,1376.016194,255.404205,579.535263,539.411255,1.5,1.0,internal_test
749,397029930331,"[[tensor(0.0006, dtype=torch.bfloat16), tensor...",397029930331,2,male,15.000000,51.0,control,1630.373420,281.434380,826.158525,521.804212,3.0,1.0,internal_test
750,707185386079,"[[tensor(0.0034, dtype=torch.bfloat16), tensor...",707185386079,8,female,24.000000,55.0,control,1303.754208,212.626037,631.421494,459.227206,3.0,1.0,external_test


In [10]:
class MRIDataset(Dataset):
    def __init__(self, df, mean=None, std=None):
        self.df = df
        self.data = self.df['embedding']
        self.mean = mean
        self.std = std
        self.age = self.df['age']
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        x = (self.data[idx] - self.mean) / (self.std + 1e-8)
        x = x.unsqueeze(0)
        y = self.age[idx]
        return x, y

In [11]:
random_seed = 42
dataset = MRIDataset(age_embedding_df, mean = mean, std=std)
train_size = int(0.70 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size
train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, test_size, val_size], generator=torch.Generator().manual_seed(random_seed))

In [12]:
print(len(train_dataset), len(val_dataset), len(test_dataset)) # for reference

526 114 112


In [13]:
batch_size = 32 if torch.cuda.is_available() else 4
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)
print(f'Using batch_size={batch_size}')

Using batch_size=32


In [14]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = RegCNN(input_height=pad_h, input_width=pad_w).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
print(f'Model initialize on {device}')

Model initialize on cuda


In [15]:
def train(model, train_loader, val_loader, epochs=100):
    train_losses = []
    val_losses = []
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0

        for emb, age in train_loader:
            emb, age = emb.to(device, dtype=torch.float32), age.to(device, dtype=torch.float32)

            optimizer.zero_grad()
            outputs = model(emb).squeeze()
            loss = criterion(outputs, age)

            loss.backward()
            optimizer.step()

            train_loss += loss.item() * emb.size(0)
        
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for emb, age in val_loader:
                emb, age = emb.to(device, dtype=torch.float32), age.to(device, dtype=torch.float32)

                outputs = model(emb).squeeze()
                loss = criterion(outputs, age)
                val_loss += loss.item() * emb.size(0)
        
        avg_train_loss = train_loss / len(train_loader.dataset)
        train_losses.append(avg_train_loss)
        avg_val_loss = val_loss / len(val_loader.dataset)
        val_losses.append(avg_val_loss)

        print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    
    return train_losses, val_losses, age.detach().cpu(), outputs.detach().cpu()

In [16]:
start_time = time.time()
train_losses, val_losses, age, outputs = train(model, train_loader, val_loader)
end_time = time.time()
total_time = end_time - start_time
print(f" Training Complete!")
print(f"Total time: {int(total_time // 60)}m {int(total_time % 60)}s")

Epoch [1/100] | Train Loss: 321.0091 | Val Loss: 212.0686


Epoch [2/100] | Train Loss: 182.0550 | Val Loss: 165.1675


Epoch [3/100] | Train Loss: 167.8395 | Val Loss: 163.9698


Epoch [4/100] | Train Loss: 164.1108 | Val Loss: 165.7955


Epoch [5/100] | Train Loss: 169.4779 | Val Loss: 171.8998


...

Epoch [96/100] | Train Loss: 0.0046 | Val Loss: 34.4772


Epoch [97/100] | Train Loss: 0.0070 | Val Loss: 34.5416


Epoch [98/100] | Train Loss: 0.0076 | Val Loss: 34.4985


Epoch [99/100] | Train Loss: 0.0073 | Val Loss: 34.3685


Epoch [100/100] | Train Loss: 0.0060 | Val Loss: 34.4262
 Training Complete!
Total time: 4m 20s


In [17]:
plt.figure(figsize=(10, 6))
plt.plot(train_losses, label='Training Loss', color='blue', lw=2)
plt.plot(val_losses, label='Validation Loss', color='red', lw=2, linestyle='--')
plt.title('Training vs Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('MSE Loss')
plt.legend()
plt.grid(True)
plt.savefig('/home/chyhsu/Documents/neurovfm/loss.png')
plt.show()

In [18]:
# plot predicted versus true validation age distributions
fix, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)

age_plot = age.detach().cpu().numpy() if isinstance(age, torch.Tensor) else np.asarray(age)
outputs_plot = outputs.detach().cpu().numpy() if isinstance(outputs, torch.Tensor) else np.asarray(outputs)

sns.histplot(age_plot, kde=True, color='skyblue', ax=axes[0])
axes[0].set_title("Distribution of true age")
axes[0].set_xlabel("Age")

sns.histplot(outputs_plot, kde=True, color='salmon', ax=axes[1])
axes[1].set_title("Distribution of predicted age")
axes[1].set_xlabel("Age")

plt.tight_layout()
plt.savefig('/home/chyhsu/Documents/neurovfm/recon_dist.png', dpi=300, bbox_inches='tight')
plt.show()